<a href="https://www.kaggle.com/code/mehulkumar99/spider-1-llama-8b-training-nl2sql-84-acc?scriptVersionId=321968460" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Read the detailed explanation and discussion for this notebook here: [NL2SQL Blog](https://www.kaggle.com/code/mehulkumar99/blog-question-to-sql-query)

Check out the  [Generation Notebook](https://www.kaggle.com/code/mehulkumar99/spider-1-inference-notebook-nl2sql)

Check out the  [Agent Notebook](https://www.kaggle.com/code/mehulkumar99/agentic-loop-addressing-syntax-errors)

Check out the  [RAG notebook](https://www.kaggle.com/code/mehulkumar99/rag-sql)

Links also present in the [NL2SQL Blog](https://www.kaggle.com/code/mehulkumar99/blog-question-to-sql-query)


## Ablation table (Accuracy Progression)

| Step | Change | Execution Accuracy |
|---|---|---|
| Phi-2 2.7B baseline | Unaugmented schema | 40% |
| Llama 3.1 8B | Unaugmented schema | 70.89% |
| + Column order fix | Normalise result tuple order | 72% |
| + Augmented inference | Sample rows, 142 examples | 75% |
| + Token count fix | All 301 wrong/error examples | 77.27% |
| + Augmented fine-tuning | 6,607 training examples | 81% |
| + Agentic loop + CoT | Chain-of-thought + retry on all failures | **84%** |


# NL2SQL Fine-Tuning Pipeline

## Overview
Fine-tunes **Llama 3.1 8B** using **QLoRA** on the Spider dataset with augmented schemas to improve natural language to SQL translation accuracy.

---

## Data Flow
The training pipeline processes data through the following stages:

1. **Spider Dataset** The baseline text-to-SQL dataset.
2. **Schema Lookup (`format_schema()`)** Augments the schema by injecting column types, 3 sample rows, and primary key/foreign key (PK/FK) annotations.
3. **Prompt Construction (`build_prompt()`)** Formats the data into the Llama 3.1 chat template and applies response-only masking.
4. **Model Training (`SFTTrainer`)** Executes QLoRA fine-tuning with a cosine learning rate scheduler and early stopping.
5. **Optimal Checkpoint (`checkpoint-X`)** Yields the best model state with an `lowest eval_loss` .

---

## Key Design Decisions
* **Augmented Schema:** Includes explicit column types and 3 sample rows per column to provide better generation context.
* **Context Window Management:** Filtered out 393 training examples that exceeded the 2,048 token limit.
* **Targeted Masking (`train_on_responses_only`):** Computes loss exclusively on the generated SQL output rather than the prompt instructions.
* **LoRA Hyperparameters:** * `r` = 16
    * `lora_alpha` = 16
    * `dropout` = 0.05
    * `weight_decay` = 0.01
    * `lr` = 5e-5

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# # Let Kaggle's existing torch stay untouched
# # Just install unsloth against whatever torch is already there
# !pip install unsloth -q
# !pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo -q

In [ ]:
!wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -


In [ ]:
import subprocess
import sys
import os

# List all your required packages here
packages = [
    "bitsandbytes", 
    "transformers", 
    "accelerate", 
    "peft", 
    "trl==0.15.2", 
    "datasets", 
    "pandas", 
    "matplotlib", 
    "seaborn", 
    "hf_transfer"
]

# Install everything using standard Python logic
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

# unsloth installed separately — no-deps to prevent it pulling a conflicting torch
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-cache-dir", "--no-deps",
    "unsloth", "unsloth_zoo"
])

# Enable high-speed Hugging Face transfers
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "freeze"
], stdout=open("requirements_frozen.txt", "w"))

In [ ]:
import os
import torch
import gc

# The fix you requested
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"


def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    print("VRAM Cleared and Memory Management Updated.")

clear_vram()

In [ ]:
import torch
import transformers
# import bitsandbytes as bnb
import peft

print(f"transformers : {transformers.__version__}")
# print(f"bitsandbytes : {bnb.__version__}")
print(f"peft         : {peft.__version__}")
print(f"torch        : {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name     : {torch.cuda.get_device_name(0)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer # Autokenizer is a class of the transformers library as it starts with capital letter.
"""Autotokenizer is a class in transformers library responsible for tokenization. """

# HANDLING THE TRAINING AND VALIDATION EXAMPLES

In [ ]:
from datasets import load_dataset #loaded into the cpu. # dataset is a library and load_dataset is a function (as it has lowercase with underscore)

dataset = load_dataset("spider")
print(dataset)
# Train: 7000 examples, Validation: 1034 examples

In [ ]:
ex = dataset['validation'][0]
print("DB ID:", ex['db_id'])
print("\nQUESTION:", ex['question'])
print("\nQUERY:", ex['query'])
print("\nQUERY TOKS:", ex['query_toks'])
print("\nQUERY TOKS NO VALUE:", ex['query_toks_no_value'])
print("\nQUESTION TOKS:", ex['question_toks'])

In [ ]:
print(dataset['validation'][0].items())

In [ ]:
validation_df = pd.DataFrame(dataset['validation'])
validation_df.head(10)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('dark_background')

train_df = pd.DataFrame(dataset['train'])

# Count SQL keywords as a complexity proxy
keywords = ['JOIN', 'GROUP BY', 'ORDER BY', 'HAVING', 'INTERSECT', 'UNION', 'EXCEPT', 'LIMIT', 'NESTED','END AS', 'CASE']

for kw in keywords:
    train_df[kw] = train_df['query'].str.upper().str.contains(kw) # use of vectorization, means the operations are done to all the rows at once

train_df['#joins'] = train_df['query'].str.upper().str.split().apply(lambda x: x.count('JOIN'))
keyword_counts = train_df[keywords].sum().sort_values(ascending=False)
print(type(keyword_counts))
print(keyword_counts)

keyword_counts.plot(kind='bar', title='SQL Clause Distribution in Training Set')
plt.tight_layout()
plt.show()

In [ ]:
# understanding the distribution of joins
joins = train_df['#joins'].tolist()



# Create the histogram
plt.hist(joins, bins=100, edgecolor='black')

# Add labels and title
plt.title('Joins per example')
plt.xlabel('#Joins')
plt.ylabel('Frequency')

# Display the plot
plt.show()
print(f'mean count of joins : {np.mean(joins)}')

In [ ]:
#distribution of keywords in validation set
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('dark_background')

validation_df = pd.DataFrame(dataset['validation'])

# Count SQL keywords as a complexity proxy
keywords = ['JOIN', 'GROUP BY', 'ORDER BY', 'HAVING', 'INTERSECT', 'UNION', 'EXCEPT', 'LIMIT', 'NESTED','END AS', 'CASE']

for kw in keywords:
    validation_df[kw] = validation_df['query'].str.upper().str.contains(kw)

keyword_counts = validation_df[keywords].sum().sort_values(ascending=False)
print(keyword_counts)
print(type(keyword_counts))

keyword_counts.plot(kind='bar', title='SQL Clause Distribution in Validation Set')
plt.tight_layout()
plt.show()

The training and validation have similar count of the keywords showing similar distribution in both

In [ ]:
validation_df.iloc[-10]

In [ ]:
db_counts = train_df['db_id'].value_counts()
print(f"Total unique DBs: {train_df['db_id'].nunique()}")  # ~140 DBs
db_counts.head(20).plot(kind='bar', title='Top 20 DBs by training set by Question Count')
plt.tight_layout()
plt.show()

In [ ]:
db_counts = validation_df['db_id'].value_counts()
print(f"Total unique DBs: {validation_df['db_id'].nunique()}")  # ~140 DBs
db_counts.head(20).plot(kind='bar', title='Top 20 DBs in validation set by Question Count')
plt.tight_layout()
plt.show()

The database in both the training and validation set are different so that the model is not memorizing stuff and overfitting leading to better generalisation.

# FORMATTING OF THE SCHEMA

In [ ]:
#loading the schema of the dbs

spider_tables = load_dataset("richardr1126/spider-schema", split="train")
print(spider_tables)

In [ ]:
# df= pd.DataFrame(spider_tables)
# index = 58
# print(f'Database at location {index} is :{df.iloc[58,0]}')

In [ ]:
# # Verifying that the database in training and validation examples are all represented in the spider_tables

# set(df['db_id']) >= set(train_df['db_id']) | set(validation_df['db_id'])

# SCHEMA CREATION

## Schema lookup

In [ ]:
import pickle
import json

# Open the file in 'rb' (read-binary) mode
with open('/kaggle/input/datasets/mehulkumar99/augmented-schema/schema_lookup_augmented.pkl', 'rb') as f:
    schema_lookup = pickle.load(f)


# db_id = list(schema_lookup.keys())[0]
db_id = 'concert_singer'
print(db_id)
print(json.dumps(schema_lookup[db_id], indent=2, default=str))

In [ ]:
schema_lookup ={}
for row in spider_tables:
  schema_lookup[row['db_id']]= row

schema_lookup['concert_singer']

In [ ]:
# print(f'Total number of databases: {len(schema_lookup)}')
# print('concert_singer' in schema_lookup)

In [ ]:
def parse_primary_keys(pk_string):

  pk_map ={}
  # There are database full of tables but has not primary keys at all.
  if not pk_string:
    return pk_map

  for entry in pk_string.split('|'):
    entry = entry.strip()
    table, cols = entry.split(':')

    pk_map[table.strip()] =cols.strip()
  return pk_map

db = 'concert_singer'
map = parse_primary_keys(pk_string = schema_lookup[db]['Primary Keys'])
print(map)

In [ ]:
def parse_foreign_keys(fk_string):
    fk_map = {}
    words = fk_string
    fk_string = words.split()
    for i,word in enumerate(fk_string):
        if word =='equals':
            first_table = fk_string[i-3]
            second_table = fk_string[i+1]

            fk_map[(first_table, fk_string[i-1])] = (second_table, fk_string[i+3])


    return fk_map
map = parse_foreign_keys(fk_string = schema_lookup[db]['Foreign Keys'])
print(map)

In [ ]:
def format_schema(db_id):
    row = schema_lookup[db_id]
    schema = row['Schema (values (type))']
    pk_map = parse_primary_keys(row['Primary Keys'])
    fk_map = parse_foreign_keys(row['Foreign Keys'])

    schema_lines = []
    for table_info in schema.split('|'):

        table_info = table_info.strip()

        table_name, cols_info = table_info.split(':')
        table_name = table_name.strip()
        cols_info = cols_info.strip()

        # a table can have no primary key.
        pk =''
        if table_name in pk_map:
            pk = pk_map[table_name]

        col_lines = []
        for col_info in cols_info.split(','):

            col_info = col_info.strip()

            col_name = col_info[:col_info.rfind('(')].strip()
            col_type = col_info[col_info.rfind('(')+1 : col_info.rfind(')')].strip()

            # checking if foreign key exist for this column
            fk_found = False
            col_to_find = col_name


            if (table_name, col_to_find) in fk_map:
                mapping = fk_map[(table_name, col_to_find)]
                fk_found = True


            annotations = []

            annotations.append(f'{ col_type}')
            if pk and col_name == pk:
                annotations.append(' PK')
            if fk_found:
                temp = f'{mapping[0]}.{mapping[1]}'
                annotations.append(f' FK -> {temp}')

            annotations = f'- {col_name} ' +  f'({','.join(annotations)})'

            col_lines.append(annotations)
        col_lines = '\n'.join(col_lines)
        schema_lines.append(f'# Table: {table_name} \n## Columns:\n{col_lines}')
    s = '\n\n'.join(schema_lines)
    schema_lookup[db_id]['format_schema'] = s

    return s

db_id = db
s = format_schema(db_id)

print(s)

In [ ]:
# schema_table= pd.DataFrame(spider_tables)
# schema_table['format_schema'] = df['db_id'].apply(format_schema)

In [ ]:
# longest_schema = max(schema_table['format_schema'],key= len)
# approx_tokens = longest_schema.split()
# print(f'length of the longest schema separated by words is: {len(approx_tokens)}')

In [ ]:
# shortest_schema = min(schema_table['format_schema'],key= len)
# approx_tokens = shortest_schema.split()
# len(approx_tokens)

# LOADING MODEL

## Understanding Rotational Position Embedding

[RoPE](https://docs.google.com/document/d/1acdbby4rbNF8yIvz4B9exZkia4XXGKlvJoop8TNyaJw/edit?usp=sharing)

In [ ]:
import psutil

ram = psutil.virtual_memory()
print(f"Total RAM in CPU     : {ram.total/1e9:.1f} GB")
print(f"Available RAM in CPU: {ram.available/1e9:.1f} GB")
print(f"Used RAM     : {ram.used/1e9:.1f} GB")

In [ ]:
import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # should say Tesla P100
    print(torch.cuda.device_count())

Above command is for anyone who wants to finetune a LLM. Though above code, we are installing a specialised set of libraries designed for PEFT (Parameter efficient fine tuning) and Quantization.
## Core engines

* transformers by Huggingface: ultimate library for downloading and running pretrained models
* accelerate: handles the movement of training across available hardwares. It handles the movement of tensor between CPU and GPU w/o writing the device allocation code

## Memory savers
* bitsandbytes: Unlocks quantization, converts 16bit model to 4 bit weights so it can fit small GPU;s VRAM. [Process of conversion](https://docs.google.com/document/d/1KWbP5Fob9UTyesJRxd09FKyLaZT29utNNUPYHOwbGL0/edit?usp=sharing)
* peft (Parameter Efficient Fine Tuning): This is what actually creates the $A$ and $B$ low-rank matrices we talked about earlier! It freezes the 4-bit base model and injects the trainable LoRA layers

## Fine Tuning Wrapper
* trl( Transformer Reinforcement Learning): [What is wrapper and what does trl do](https://docs.google.com/document/d/1maHPmdxABUpXq38AOpel-6qO7lmBNklzoRozx9UWRro/edit?tab=t.0)

## Transformers Library

These are the absolute standard tools you need to load an LLM like Phi-2 or Llama 3 for text generation and fine-tuning.

Notice how most of these start with the word "Auto"? That goes back to what we just talked about regarding "wrappers." Instead of you having to figure out the exact math classes for Phi-2 specifically, the "Auto" classes automatically read the model's name and do the heavy lifting for you

* Autotokenizer: Hugging Face automatically downloads the exact dictionary and rules that the Phi-2 creators used to train it.[.eos_token](https://docs.google.com/document/d/1YThD3wZ93_K38LxSwDIFk4_O6SrQyn9Mo2z20Aj0dlc/edit?usp=sharing)
* [AutoModelForCausalLM](https://docs.google.com/document/d/1fo5HTByWJdmNP_WFATlUknFNpJsWGYWWaTtI4qRWWYE/edit?usp=sharing): Microsoft has trained the model to learn the embeddings of the words in the text, now we can use these embeddings to 
  * generate next word, code completion
  * output a single number or category (for sentiment analysis,or spam detection (AutoModelForSequenceClassification is used for this)
* Autoconfig: Every model has a JSON file sitting on the Hugging Face Hub called config.json. This file contains the exact blueprint of the model: how many layers it has, the dimension size of the vectors, the number of attention heads, etc.
**Why use it?** You use AutoConfig if you want to inspect or change the blueprint before loading the model. For example, if you wanted to change Phi-2's max sequence length from 2048 to something else before building it!
*  BitsAndBytesConfig: This is the configuration tool from the bitsandbytes library that we just discussed when talking about 4-bit memory!
**Why use it?** This is where you write the settings to tell the GPU, "Hey, load this massive 2.7-billion-parameter Phi-2 model, but squeeze it down into 4-bit storage so it fits in my smaller VRAM."

# LOADING LLAMA 8.1

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype        = None,
    load_in_4bit = True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r            = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha   = 16,
    lora_dropout = 0.05,
    bias         = "none",
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
    use_rslora   = False,
    loftq_config = None,
)

print(f"Memory used: {model.get_memory_footprint()/1e9:.2f} GB")
model.print_trainable_parameters()

In [ ]:


# 1. Count how many GPUs PyTorch sees
num_gpus = torch.cuda.device_count()
print(f"Total GPUs detected: {num_gpus}\n")

# 2. Loop through every detected GPU
for i in range(num_gpus):
    # Get properties of the specific GPU (like its name and total capacity)
    props = torch.cuda.get_device_properties(i)
    total_mem = props.total_memory / 1e9
    
    # Check how much memory is actively allocated right now
    allocated_mem = torch.cuda.memory_allocated(i) / 1e9
    
    # Check how much PyTorch is reserving (caching) from the OS
    reserved_mem = torch.cuda.memory_reserved(i) / 1e9
    
    print(f"--- GPU {i}: {props.name} ---")
    print(f"  Total Memory    : {total_mem:.2f} GB")
    print(f"  Allocated Memory: {allocated_mem:.2f} GB")
    print(f"  Reserved Memory : {reserved_mem:.2f} GB")
    print(f"  Available Memory: {(total_mem - allocated_mem):.2f} GB\n")

In [ ]:
# Grab the very first parameter in the model's memory
first_param = next(model.parameters())

# Print its data type
print(f"The model is actually loaded in: {first_param.dtype}")

4-bit storage → dequantize to fp16 → matmul in fp16 → output in fp16

The LoRA adapter weights are in fp16/fp32
The LoRA matrices (A and B) that you're actually training are small and kept in fp16 or fp32 — not quantized. This is fine because they're tiny (~1-2% of total parameters). Gradients flow only through these adapters, never back into the frozen 4-bit base weights.

[table displaying the above](https://docs.google.com/document/d/15SSxpjEEiwDO2bXmR9sP-HNYB3mF-ypVIlpp61oyuoc/edit?usp=sharing)

# CREATING Full text 

takes in the format schema from the spider_tables database and the query and question from the dataset to create a prompt

In [ ]:
print(f"Memory used: {model.get_memory_footprint()/1e9:.2f} GB")
print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU remaining: {(16 - torch.cuda.memory_allocated()/1e9):.2f} GB")

# CREATING Full text 

takes in the format schema from the spider_tables database and the query and question from the dataset to create a prompt

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'llama', # change this to the right chat_template name
)

In [ ]:
import pickle
import json

# Open the file in 'rb' (read-binary) mode
with open('/kaggle/input/datasets/mehulkumar99/augmented-schema/schema_lookup_augmented.pkl', 'rb') as f:
    schema_lookup = pickle.load(f)


# db_id = list(schema_lookup.keys())[0]
db_id = 'concert_singer'
print(db_id)
print(json.dumps(schema_lookup[db_id], indent=2, default=str))

In [ ]:
def build_prompt( schema_lookup, index, dataset, inference = False, schema_type = None ):

    df = dataset
    prompt = ''

    db_id = df['db_id'][index]
    question = df['question'][index]

    if not schema_type:
        schema = schema_lookup[db_id]['format_schema']
    else:
        schema = schema_lookup[db_id]['augmented_schema']
    
    prompt = f"""<|start_header_id|>system<|end_header_id|>

Convert the natural language question to SQL using the schema below. Use SQLite syntax and SQLite functions only.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Schema:
{schema}

### Question:
{question}"""

    if inference == True:
        text = prompt +  f"""<|eot_id|>""" + "<|start_header_id|>assistant<|end_header_id|>\n\n"
        return {'text': text}


    elif inference == False:
        query = df['query'][index]
        text = prompt+ f"""<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{query}<|eot_id|>"""
        
        return {'text':text}
    return{'text': prompt}



text = build_prompt( schema_lookup, index =779,dataset = validation_df, inference = True, schema_type = 1 )
print(text['text'])
    

In [ ]:
#schema_lookup, index, dataset, last_exec_sql = None, error_msg = None, inference = False 
total = len(train_df)
lengths = []
for i in range(total):
    prompt = build_prompt( schema_lookup, index = i , dataset = train_df, inference = False, schema_type = 1 )['text']
    lengths.append(len(tokenizer(prompt)['input_ids']))

train_df['T'] = lengths
train_df = train_df.drop(columns = ['JOIN', 'GROUP BY', 'ORDER BY', 'HAVING', 'INTERSECT', 'UNION', 'EXCEPT', 'LIMIT', 'NESTED','END AS', 'CASE'])

train_df = train_df.sort_values(by='T', ascending = False)
# & (validation_df['T']>1950)
filtered_df = train_df[(train_df['T']>2048) ]

# wrong_erro_df = filtered_df.loc[filtered_df['results'].isin(['error', 'wrong'])]
# filtered_sorted_df = filtered_df.sort_values(by='T', ascending = False)

# print(len(idx_prompt_limit_exceeded))
filtered_df.head()

In [ ]:
len(filtered_df)

In [ ]:
train_filtered_df = train_df[train_df['T'] <= 2048].reset_index(drop=True)
print(f"Training examples: {len(train_filtered_df)}")

In [ ]:
# def build_prompt( schema_lookup, index, dataset, inference = False ):

#     df = dataset
#     prompt = ''

#     db_id = df['db_id'][index]
#     question = df['question'][index]
#     schema = schema_lookup[db_id]['format_schema']
    
#     prompt = f"""<|start_header_id|>system<|end_header_id|>

# Convert the natural language question to SQL using the schema below. Use SQLite syntax and SQLite functions only.{tokenizer.eos_token}<|start_header_id|>user<|end_header_id|>

# ### Schema:
# {schema}

# ### Question:
# {question + tokenizer.eos_token}"""

#     if inference == False:
#         query = df['query'][index]
#         text = prompt+ f"""<|start_header_id|>assistant<|end_header_id|>

# {query + tokenizer.eos_token}"""
        
#         return {'text':text}
#     return{'text': prompt}
    

# train_df = pd.DataFrame(dataset['train'])
# validation_df = pd.DataFrame(dataset['validation'])

# text = build_prompt( schema_lookup, index =0,dataset = validation_df, inference = False )
# print(text)

# Train and Validation set

## Deleting longer text from train_dataset

In [ ]:
from datasets import Dataset

def build_all_prompts(df, schema_lookup):
    texts = []
    skipped = 0
    for i in range(len(df)):
        result = build_prompt(schema_lookup, i, df, inference=False, schema_type =1)
        tokens = tokenizer(
            result['text'],
            truncation=False,
            padding=False,
            add_special_tokens=False
        )
        if len(tokens['input_ids']) <= 2048:
            texts.append(result)
        else:
            skipped += 1
    print(f"Skipped {skipped} examples over 2048 tokens")
    return Dataset.from_list(texts)

train_dataset = build_all_prompts(train_filtered_df, schema_lookup)
val_dataset   = build_all_prompts(validation_df, schema_lookup)

In [ ]:
tokenizer('i love you')

In [ ]:
# print(f'eos_token is: {tokenizer.eos_token}')

# # Understanding tokeniztion, padding, end of sentence, labels managements

# tokenizer.pad_token =  "<|finetune_right_pad_id|>"
# tokenizer.padding_side = 'right'

# A = tokenizer(['I am the god of the new world', 'We love you'],
#              padding = True,
#              return_tensors = 'pt'
#              )
# print(A)



# token_id = tokenizer.eos_token_id
# print(token_id)
# token_string = tokenizer.decode(token_id)
# print(token_string)



# prompt = ['select * from students', 'select name from teachers where id =1']

# encoded_output = tokenizer( prompt,
#              padding = True,
#              return_tensors = 'pt'
#              )
# print(encoded_output)
# print('-----------------------------------')

# for i, query in enumerate(prompt):
#     prompt[i]+= tokenizer.eos_token

# encoded_output = tokenizer( prompt,
#              padding = True,
#              return_tensors = 'pt'
#              )
# print(encoded_output)

# TRAINING ARGUMENTS

In [ ]:
from trl import SFTTrainer, SFTConfig


training_args=SFTConfig(
    output_dir="./outputs",
    weight_decay = 0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size = 4,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    num_train_epochs=1,
    learning_rate=2e-5,
    fp16=True,                  # T4 doesn't support bf16
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    optim="paged_adamw_8bit",         # saves memory vs full adamw
    seed=42,
    report_to="none",           # set to "wandb" if you want tracking
    max_seq_length=2048,
    dataset_text_field="text",  # the field in your dataset that has the formatted prompt
    disable_tqdm = False,
    lr_scheduler_type="cosine",
    load_best_model_at_end = True ,       # Tells the trainer to track the "Best" models
    metric_for_best_model = 'eval_loss',  # The metric to decide what is "best"
    greater_is_better = False,             # False because we want the LOWEST loss
    save_only_model=True,
    gradient_checkpointing = True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # needed for unsloth
    
)

In [ ]:
len(train_dataset)

In [ ]:

sample = train_dataset[0]['text']
print(sample[:500])


In [ ]:
from unsloth.chat_templates import train_on_responses_only
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args= training_args,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=10)], # stop if eval_loss doesn't improve for 3 evals (3*eval_steps = 3*50 = 150.)
)

# Apply response-only masking after trainer is constructed
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
# To release stale memory
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Directly inspect the first example's labels after collation
sample = train_dataset[0]  # definitely the first example

# Manually tokenize and collate it
inputs = tokenizer(
    sample['text'],
    return_tensors='pt',
    truncation=True,
    max_length=2048,
    add_special_tokens=False
)

# Check where -100 masking ends in the labels
# train_on_responses_only works on the collated batch labels
# so use the dataloader but fix the seed first
import torch
torch.manual_seed(0)
batch = next(iter(trainer.get_train_dataloader()))
labels    = batch['labels'][0]
input_ids = batch['input_ids'][0]

# Find first non -100 position
first_real = (labels != -100).nonzero()[0].item()
print("Token before masking ends:", tokenizer.decode(input_ids[first_real - 1]))
print("First token with real loss:", tokenizer.decode(input_ids[first_real]))

# Also print a window around the boundary
print("\nBoundary context:")
print(tokenizer.decode(input_ids[first_real-5 : first_real+5]))

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
train_output = trainer.train(resume_from_checkpoint=True)
print(train_output)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract logs from trainer
logs = trainer.state.log_history

# Separate train and val logs
train_logs = [l for l in logs if 'loss' in l and 'eval_loss' not in l]
eval_logs  = [l for l in logs if 'eval_loss' in l]

train_steps = [l['step'] for l in train_logs]
train_loss  = [l['loss'] for l in train_logs]

eval_steps  = [l['step'] for l in eval_logs]
eval_loss   = [l['eval_loss'] for l in eval_logs]

# Plot
plt.figure(figsize=(12, 5))
plt.plot(train_steps, train_loss, label='Train Loss', marker='o')
plt.plot(eval_steps,  eval_loss,  label='Val Loss',   marker='o')
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/loss_curve.png")
plt.show()

In [ ]:
# Add a rolling mean to train loss to see the trend under the noise
import pandas as pd
train_steps  = [x['step'] for x in trainer.state.log_history if 'loss' in x]
train_losses = [x['loss'] for x in trainer.state.log_history if 'loss' in x]

rolling_mean = pd.Series(train_losses).rolling(window=20).mean()

plt.plot(train_steps, train_losses, alpha=0.3, label='Train Loss (raw)')
plt.plot(train_steps, rolling_mean, label='Train Loss (smoothed)', linewidth=2)

## Checkpoints

In [ ]:
# Sort eval logs by loss
eval_df = pd.DataFrame(eval_logs)[['step', 'eval_loss']]
eval_df = eval_df.sort_values('eval_loss')
print(eval_df.head(10))

# Top 3 best checkpoints
top3_steps = eval_df.head(3)['step'].tolist()
print(f"\nTop 3 checkpoints: {top3_steps}")
print(f"Best checkpoint  : step {top3_steps[0]}")

In [ ]:
import os

checkpoints = [d for d in os.listdir("/kaggle/working/outputs/") if "checkpoint-" in d]
print(f"The surviving Top-3 checkpoints are: {checkpoints}")

In [ ]:
import pickle
import pandas as pd

import os
os.makedirs('/kaggle/working/saved_data', exist_ok=True)

train_filtered_df.to_pickle('/kaggle/working/saved_data/train_filtered_df.pkl')